In [1]:
import pandas as pd

In [2]:
events = {
    "event_id": [
        "E001", "E002", "E003", "E004", "E005",
        "E006", "E007", "E008", "E009", "E010"
    ],
    "device_id": [
        "D101", "D101", "D102", "D103", "D102",
        "D104", "D101", "D103", "D104", "D105"
    ],
    "event_time": [
        "2026-09-20 08:10:00",
        "2026-09-20 09:45:00",
        "2026-09-20 10:20:00",
        "invalid",
        "2026-09-20 14:30:00",
        "2026-09-20 18:15:00",
        "2026-09-21 07:50:00",
        "2026-09-21 08:30:00",
        "2026-09-21 09:10:00",
        "2026-09-21 10:00:00"
    ],
    "ingestion_time": [
        "2026-09-20 08:12:00",
        "2026-09-20 09:50:00",
        "2026-09-20 10:50:00",
        "2026-09-20 12:00:00",
        "2026-09-20 14:32:00",
        "2026-09-20 18:45:00",
        "2026-09-21 07:55:00",
        "2026-09-21 08:32:00",
        "2026-09-21 10:00:00",
        "2026-09-21 10:03:00"
    ],
    "temperature": [
        45, 52, 71, 60, 68,
        75, 49, 82, 65, 58
    ]
}

df = pd.DataFrame(events)

### Inspection

In [3]:
df.head()

,event_id,device_id,event_time,ingestion_time,temperature
0,E001,D101,2026-09-20 08:10:00,2026-09-20 08:12:00,45
1,E002,D101,2026-09-20 09:45:00,2026-09-20 09:50:00,52
2,E003,D102,2026-09-20 10:20:00,2026-09-20 10:50:00,71
3,E004,D103,invalid,2026-09-20 12:00:00,60
4,E005,D102,2026-09-20 14:30:00,2026-09-20 14:32:00,68


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   event_id        10 non-null     str  
 1   device_id       10 non-null     str  
 2   event_time      10 non-null     str  
 3   ingestion_time  10 non-null     str  
 4   temperature     10 non-null     int64
dtypes: int64(1), str(4)
memory usage: 532.0 bytes


Parse timestamps

- Convert event_time and ingestion_time to datetime.
- Invalid timestamps must not crash the pipeline.

In [5]:
df['event_time'] = pd.to_datetime(df['event_time'], errors='coerce')

In [6]:
df['ingestion_time'] = pd.to_datetime(df['ingestion_time'], errors='coerce')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   event_id        10 non-null     str           
 1   device_id       10 non-null     str           
 2   event_time      9 non-null      datetime64[us]
 3   ingestion_time  10 non-null     datetime64[us]
 4   temperature     10 non-null     int64         
dtypes: datetime64[us](2), int64(1), str(2)
memory usage: 532.0 bytes


In [8]:
df.head()

,event_id,device_id,event_time,ingestion_time,temperature
0,E001,D101,2026-09-20 08:10:00,2026-09-20 08:12:00,45
1,E002,D101,2026-09-20 09:45:00,2026-09-20 09:50:00,52
2,E003,D102,2026-09-20 10:20:00,2026-09-20 10:50:00,71
3,E004,D103,NaT,2026-09-20 12:00:00,60
4,E005,D102,2026-09-20 14:30:00,2026-09-20 14:32:00,68


Data-quality check

- Identify records with invalid/missing event_time.
- Create a separate DataFrame called invalid_events.

In [9]:
invalid_events = df[df['event_time'].isna()]

In [10]:
invalid_events

,event_id,device_id,event_time,ingestion_time,temperature
3,E004,D103,NaT,2026-09-20 12:00:00,60


Create the valid dataset

Create valid_events containing only records with valid event timestamps.

In [11]:
valid_events = df[~df['event_time'].isna()]

In [12]:
valid_events

,event_id,device_id,event_time,ingestion_time,temperature
0,E001,D101,2026-09-20 08:10:00,2026-09-20 08:12:00,45
1,E002,D101,2026-09-20 09:45:00,2026-09-20 09:50:00,52
2,E003,D102,2026-09-20 10:20:00,2026-09-20 10:50:00,71
4,E005,D102,2026-09-20 14:30:00,2026-09-20 14:32:00,68
5,E006,D104,2026-09-20 18:15:00,2026-09-20 18:45:00,75
6,E007,D101,2026-09-21 07:50:00,2026-09-21 07:55:00,49
7,E008,D103,2026-09-21 08:30:00,2026-09-21 08:32:00,82
8,E009,D104,2026-09-21 09:10:00,2026-09-21 10:00:00,65
9,E010,D105,2026-09-21 10:00:00,2026-09-21 10:03:00,58


Calculate ingestion delay

- Create ingestion_delay_minutes.
- It represents the number of minutes between the event occurring and being ingested.

In [17]:
valid_events['ingestion_delay_minutes'] = (valid_events['ingestion_time'] - valid_events['event_time']).dt.total_seconds() / 60

In [18]:
valid_events

,event_id,device_id,event_time,ingestion_time,temperature,ingestion_delay_minutes
0,E001,D101,2026-09-20 08:10:00,2026-09-20 08:12:00,45,2.0
1,E002,D101,2026-09-20 09:45:00,2026-09-20 09:50:00,52,5.0
2,E003,D102,2026-09-20 10:20:00,2026-09-20 10:50:00,71,30.0
4,E005,D102,2026-09-20 14:30:00,2026-09-20 14:32:00,68,2.0
5,E006,D104,2026-09-20 18:15:00,2026-09-20 18:45:00,75,30.0
6,E007,D101,2026-09-21 07:50:00,2026-09-21 07:55:00,49,5.0
7,E008,D103,2026-09-21 08:30:00,2026-09-21 08:32:00,82,2.0
8,E009,D104,2026-09-21 09:10:00,2026-09-21 10:00:00,65,50.0
9,E010,D105,2026-09-21 10:00:00,2026-09-21 10:03:00,58,3.0


Create time attributes

- Add event_date.
- Add event_hour.

In [20]:
valid_events['event_date'] = valid_events['event_time'].dt.date

In [22]:
valid_events['event_hour'] = valid_events['event_time'].dt.hour

In [23]:
valid_events

,event_id,device_id,event_time,ingestion_time,temperature,ingestion_delay_minutes,event_date,event_hour
0,E001,D101,2026-09-20 08:10:00,2026-09-20 08:12:00,45,2.0,2026-09-20,8
1,E002,D101,2026-09-20 09:45:00,2026-09-20 09:50:00,52,5.0,2026-09-20,9
2,E003,D102,2026-09-20 10:20:00,2026-09-20 10:50:00,71,30.0,2026-09-20,10
4,E005,D102,2026-09-20 14:30:00,2026-09-20 14:32:00,68,2.0,2026-09-20,14
5,E006,D104,2026-09-20 18:15:00,2026-09-20 18:45:00,75,30.0,2026-09-20,18
6,E007,D101,2026-09-21 07:50:00,2026-09-21 07:55:00,49,5.0,2026-09-21,7
7,E008,D103,2026-09-21 08:30:00,2026-09-21 08:32:00,82,2.0,2026-09-21,8
8,E009,D104,2026-09-21 09:10:00,2026-09-21 10:00:00,65,50.0,2026-09-21,9
9,E010,D105,2026-09-21 10:00:00,2026-09-21 10:03:00,58,3.0,2026-09-21,10


Incremental processing window

- Assume this pipeline run processes:
- 2026-09-21 08:00:00 <= event_time < 2026-09-21 10:00:00

In [27]:
current_batch = valid_events.loc[
    (valid_events['event_time'] >= "2026-09-21 08:00:00") & 
    (valid_events['event_time'] < "2026-09-21 10:00:00"),
    :
]

In [28]:
current_batch

,event_id,device_id,event_time,ingestion_time,temperature,ingestion_delay_minutes,event_date,event_hour
7,E008,D103,2026-09-21 08:30:00,2026-09-21 08:32:00,82,2.0,2026-09-21,8
8,E009,D104,2026-09-21 09:10:00,2026-09-21 10:00:00,65,50.0,2026-09-21,9


Device-level KPI

- Using valid_events, produce one row per device_id containing:

- device_id, event_count, avg_temperature, max_temperature, avg_ingestion_delay_minutes

In [33]:
valid_events.groupby('device_id', as_index=False).agg(
    event_count=('event_id', 'count'),
    avg_temperature=('temperature', 'mean'),
    max_temperature=('temperature', 'max'),
    avg_ingestion_delay_minutes=('ingestion_delay_minutes', 'mean')
)

,device_id,event_count,avg_temperature,max_temperature,avg_ingestion_delay_minutes
0,D101,3,48.666667,52,4.0
1,D102,2,69.500000,71,16.0
2,D103,1,82.000000,82,2.0
3,D104,2,70.000000,75,40.0
4,D105,1,58.000000,58,3.0
